# Avance 1 – EDA de la tabla de modelado supervisado | ML–SEDP
**Unidad:** sitio de aforo SICT (TOP) · **Variable objetivo:** `log_tdpa` = ln(TDPA en ambos sentidos)
· **Conjunto de predicción:** CTN del corredor MEX-095D

**Planteamiento.** El modelo aprende la relación entre el entorno territorial de un tramo (vía, población,
economía, clima, agricultura) y su tránsito observado, para **estimar el TDPA con incertidumbre en los CTN
donde no hay estación de aforo**. El TDPA es observado por SICT (clase O) y **no se construye a partir de las
entradas**, por lo que no es una etiqueta de "sitio ideal" (guía ML–SEDP §9). La demanda estimada es un insumo
para caracterizar nodos Green Stop, no una decisión de construcción.

**Validación prevista:** validación cruzada espacial agrupada (por bloques de 25 km / carretera) + prueba externa
con los sitios sobre la MEX-095D, que nunca se usan para entrenar.

In [ ]:
from pathlib import Path
import warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120, "display.width", 200)
ROOT = Path.cwd().parent if Path.cwd().name == "04_NOTEBOOKS" else Path.cwd()
P, I, D = ROOT / "03_PROCESSED", ROOT / "02_INTERIM", ROOT / "00_DOCUMENTATION"
FIG = ROOT / "06_RESULTS" / "figures_top"; FIG.mkdir(parents=True, exist_ok=True)
def savefig(n): plt.tight_layout(); plt.savefig(FIG / f"{n}.png", dpi=150, bbox_inches="tight"); plt.show()

df = pd.read_csv(P / "TOP_Model_Table.csv", dtype={"cve_mun": str, "group_state": str})
feats = pd.read_csv(P / "model_feature_list.csv")["feature"].tolist()
prov = pd.read_csv(D / "feature_provenance.csv")
ctn = pd.read_csv(P / "CTN_Prediction_Table.csv") if (P / "CTN_Prediction_Table.csv").exists() else None
TARGET = "log_tdpa"
train = df[~df["is_corridor_test"]].copy(); test = df[df["is_corridor_test"]].copy()
print(f"Sitios: {len(df)} | entrenamiento/validación: {len(train)} | prueba externa MEX-095D: {len(test)}")
print(f"Features candidatas: {len(feats)}")

## 1. Estructura de los datos
### 1.1 Forma, tipos y rol de cada columna

In [ ]:
num = [c for c in feats if pd.api.types.is_numeric_dtype(df[c]) and df[c].dtype != bool]
cat = [c for c in feats if c not in num]
roles = pd.DataFrame({"dtype": df.dtypes.astype(str), "n_unique": df.nunique(),
                      "rol": ["objetivo" if c in ("tdpa", TARGET) else "feature numérica" if c in num
                              else "feature categórica" if c in cat else "metadato/agrupación" for c in df.columns]})
roles = roles.join(prov.set_index("feature_name")[["class", "source", "source_period", "spatial_resolution", "leakage_risk"]])
print(roles["rol"].value_counts().to_string()); roles

### 1.2 Estadísticas descriptivas

In [ ]:
desc = df[["tdpa", TARGET] + num].describe(percentiles=[.05, .25, .5, .75, .95]).T
desc["skew"] = df[["tdpa", TARGET] + num].skew(); desc["cv"] = desc["std"] / desc["mean"].abs()
desc.round(3)

### 1.3 Variables categóricas: frecuencias y cardinalidad

In [ ]:
for c in cat + ["group_state", "red"]:
    if c in df:
        vc = df[c].astype(str).value_counts(dropna=False)
        print(f"\n{c} (cardinalidad={df[c].nunique()})"); print((vc.to_frame("n").assign(pct=lambda x: (100*x.n/len(df)).round(1))).head(15).to_string())
print("\nAgrupaciones para validación espacial:")
print({g: df[g].nunique() for g in ["group_state", "group_route", "group_block25"]})
for name, f in {"DENUE Clase_actividad": "denue_establecimientos_sites.csv", "SIAP cultivo": "dgsiap_cultivos_municipios_corredor_sites.csv"}.items():
    if (I / f).exists():
        t = pd.read_csv(I / f, low_memory=False); col = "Clase_actividad" if "DENUE" in name else "cultivo"
        print(f"{name}: cardinalidad en origen = {t[col].nunique()} ({len(t)} registros)")

### 1.4 Valores faltantes y patrones de ausencia

In [ ]:
miss = df[feats].isna().mean().mul(100).round(1).sort_values(ascending=False)
miss = miss[miss > 0].to_frame("% faltante")
display(miss.join(prov.set_index("feature_name")[["source", "spatial_resolution"]]))
if len(miss):
    fig, ax = plt.subplots(figsize=(12, 5))
    order = df.sort_values(["group_state", "lon"]).index
    sns.heatmap(df.loc[order, miss.index].isna().T, cbar=False, cmap="Greys", ax=ax, xticklabels=False)
    ax.set_title("Faltantes por sitio (ordenados por estado y longitud)"); savefig("01_missing_pattern")
    # ¿La ausencia depende del objetivo? (si sí, imputar sin cuidado sesga el modelo)
    rows = []
    for c in miss.index:
        a, b = df.loc[df[c].isna(), TARGET], df.loc[df[c].notna(), TARGET]
        if len(a) > 2 and len(b) > 2:
            rows.append([c, a.median(), b.median(), stats.mannwhitneyu(a, b).pvalue])
    display(pd.DataFrame(rows, columns=["feature", "mediana log_tdpa si falta", "si no falta", "p_value"]).round(4))

## 2. Variable objetivo
### 2.1 Distribución, sesgo y transformación

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df["tdpa"], kde=True, ax=ax[0]); ax[0].set_title(f"TDPA (skew={df['tdpa'].skew():.2f})")
sns.histplot(df[TARGET], kde=True, ax=ax[1], color="#2a7f8f"); ax[1].set_title(f"ln(TDPA) (skew={df[TARGET].skew():.2f})")
stats.probplot(df[TARGET], dist="norm", plot=ax[2]); ax[2].set_title("Q-Q ln(TDPA)")
savefig("02_target_distribution")
print("Shapiro-Wilk ln(TDPA): p =", round(stats.shapiro(df[TARGET]).pvalue, 4))
q1, q3 = df[TARGET].quantile([.25, .75]); iqr = q3 - q1
out = df[(df[TARGET] < q1 - 1.5*iqr) | (df[TARGET] > q3 + 1.5*iqr)]
print(f"Atípicos del objetivo (IQR, escala log): {len(out)}")
display(out[["site_id", "carretera", "lugar_base", "tdpa", "te_list"]])

### 2.2 ¿Desequilibrio? Distribución del objetivo por grupos
Es regresión, así que no hay clases; el equivalente es revisar si algunos grupos (tipo de red, cuota, estado)
están sobre o subrepresentados, porque eso afecta la validación agrupada y la transferencia al corredor.

In [ ]:
by = [c for c in ["red", "toll_road", "group_state"] if c in df]
fig, ax = plt.subplots(1, len(by), figsize=(6*len(by), 4), squeeze=False)
for a, b in zip(ax.flat, by):
    sns.boxplot(data=df, x=b, y=TARGET, ax=a, color="#9bc4cb"); sns.stripplot(data=df, x=b, y=TARGET, ax=a, color="k", size=2)
    groups = [g[TARGET] for _, g in df.groupby(b) if len(g) > 1]
    p = stats.kruskal(*groups).pvalue if len(groups) > 1 else np.nan
    a.set_title(f"ln(TDPA) por {b} (Kruskal p={p:.3g})"); a.tick_params(axis="x", rotation=30)
savefig("03_target_by_group")
display(df.groupby("toll_road")[TARGET].agg(["count", "mean", "median"]).round(2))

## 3. Análisis univariante de las features

In [ ]:
show = [c for c in num if not c.endswith(("_min_month", "_missing_pct"))]
n = len(show); nc = 4; nr = int(np.ceil(n / nc))
fig, axes = plt.subplots(nr, nc, figsize=(16, 3*nr))
for a, c in zip(axes.flat, show):
    sns.histplot(df[c].dropna(), ax=a, color="#2a7f8f"); a.set_title(f"{c}\nskew={df[c].skew():.2f}", fontsize=8); a.set_xlabel("")
for a in axes.flat[n:]: a.axis("off")
savefig("04_hist_features")
fig, axes = plt.subplots(nr, nc, figsize=(16, 2.4*nr))
for a, c in zip(axes.flat, show):
    sns.boxplot(x=df[c], ax=a, color="#c9a227"); a.set_title(c, fontsize=8); a.set_xlabel("")
for a in axes.flat[n:]: a.axis("off")
savefig("05_box_features")
sk = pd.DataFrame({"skew": df[show].skew()})
pos = [c for c in show if (df[c].dropna() >= 0).all()]
sk.loc[pos, "skew_log1p"] = np.log1p(df[pos]).skew()
SKEWED = sk.index[(sk["skew"].abs() > 1) & (sk["skew_log1p"].abs() < sk["skew"].abs())].tolist()
print("Candidatas a log1p:", SKEWED); sk.round(2).sort_values("skew", ascending=False)

In [ ]:
plot_cat = [c for c in cat if df[c].nunique() <= 25]
if plot_cat:
    fig, ax = plt.subplots(1, len(plot_cat), figsize=(6*len(plot_cat), 4), squeeze=False)
    for a, c in zip(ax.flat, plot_cat):
        df[c].astype(str).value_counts().plot.barh(ax=a, color="#4f7f3a"); a.set_title(c)
    savefig("06_bar_categorical")

## 4. Dimensión temporal

In [ ]:
print("Años de aforo:", df["anio"].value_counts().to_dict(), "-> un solo año: no hay mezcla temporal en el objetivo")
f = I / "nasa_power_monthly_sites.csv"
if f.exists():
    nm = pd.read_csv(f); clim = nm.groupby(["parameter", "month"])["value"].mean().unstack(0)
    clim.plot(subplots=True, layout=(1, clim.shape[1]), figsize=(4*clim.shape[1], 3), marker="o", legend=False, title=list(clim.columns))
    savefig("07_nasa_seasonality")

## 5. Análisis bi/multivariante
### 5.1 Correlación de cada feature con el objetivo

In [ ]:
rel = pd.DataFrame({c: stats.spearmanr(df[c], df[TARGET], nan_policy="omit")[:2] for c in num}, index=["rho", "p"]).T
rel["abs"] = rel["rho"].abs(); rel = rel.sort_values("abs", ascending=False)
r_crit = np.tanh(1.96 / np.sqrt(len(df) - 3))
fig, ax = plt.subplots(figsize=(8, max(4, .25*len(rel))))
rel["rho"].iloc[::-1].plot.barh(ax=ax, color=np.where(rel["rho"].iloc[::-1] > 0, "#2a7f8f", "#c9a227"))
ax.axvline(r_crit, ls="--", c="gray"); ax.axvline(-r_crit, ls="--", c="gray")
ax.set_title(f"Spearman feature vs ln(TDPA) (líneas: |rho| crítico={r_crit:.2f})"); savefig("08_corr_with_target")
display(rel.round(3).head(20))

> **Advertencia de leakage:** este ranking es descriptivo. Si se usa para *seleccionar* features, la selección debe
> repetirse dentro de cada fold de entrenamiento; seleccionar con todos los datos y luego validar infla las métricas.

In [ ]:
top = rel.index[:6].tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for a, c in zip(axes.flat, top):
    x = np.log1p(df[c].clip(lower=0)) if c in SKEWED else df[c]
    sns.regplot(x=x, y=df[TARGET], ax=a, scatter_kws={"s": 12, "alpha": .6}, line_kws={"color": "r"})
    a.set_xlabel(("log1p " if c in SKEWED else "") + c)
savefig("09_scatter_top_features")

### 5.2 Correlación entre features (multicolinealidad)

In [ ]:
sp = df[num].corr(method="spearman")
fig, ax = plt.subplots(figsize=(14, 12)); sns.heatmap(sp, cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax, cbar_kws={"shrink": .6})
ax.set_title("Spearman entre features"); savefig("10_corr_features")
pairs = sp.where(np.triu(np.ones(sp.shape), 1).astype(bool)).stack().rename("rho").to_frame()
pairs["abs"] = pairs["rho"].abs(); display(pairs.sort_values("abs", ascending=False).head(15).round(3))

### 5.3 Autocorrelación espacial del objetivo
Comparamos el ln(TDPA) de cada sitio con el de su vecino más cercano según distancia. Si sitios cercanos se parecen
mucho, una partición aleatoria pondría casi el mismo dato en entrenamiento y prueba. La distancia a la que la
correlación cae sugiere el tamaño de bloque para la validación espacial.

In [ ]:
import geopandas as gpd
g = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326").to_crs("EPSG:6372")
xy = np.c_[g.geometry.x, g.geometry.y]; dmat = np.hypot(xy[:, None, 0] - xy[None, :, 0], xy[:, None, 1] - xy[None, :, 1])
np.fill_diagonal(dmat, np.inf); y = df[TARGET].values
bins = [0, 2e3, 5e3, 10e3, 25e3, 50e3, 100e3]; res = []
for lo, hi in zip(bins[:-1], bins[1:]):
    i, j = np.where((dmat > lo) & (dmat <= hi)); m = i < j
    if m.sum() > 10:
        res.append([f"{lo/1e3:.0f}-{hi/1e3:.0f} km", m.sum(), stats.spearmanr(y[i[m]], y[j[m]])[0],
                    0.5 * np.mean((y[i[m]] - y[j[m]])**2)])
ac = pd.DataFrame(res, columns=["distancia", "pares", "spearman", "semivarianza"]); display(ac.round(3))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(ac["distancia"], ac["spearman"], marker="o"); ax[0].set_title("Correlación ln(TDPA) entre pares por distancia")
ax[1].plot(ac["distancia"], ac["semivarianza"], marker="o", c="#c9a227"); ax[1].axhline(np.var(y), ls="--", c="gray")
ax[1].set_title("Semivariograma empírico (línea: varianza total)")
for a in ax: a.tick_params(axis="x", rotation=30)
savefig("11_spatial_autocorrelation")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(df.lon, df.lat, c=df[TARGET], cmap="viridis", s=25)
ax.scatter(test.lon, test.lat, facecolors="none", edgecolors="red", s=80, label="prueba externa MEX-095D")
plt.colorbar(sc, label="ln(TDPA)"); ax.legend(); ax.set_title("Sitios de aforo y objetivo"); savefig("12_map_target")

### 5.4 ¿Los CTN se parecen a los sitios de entrenamiento? (dominio de aplicabilidad)
Si los CTN tienen valores fuera del rango visto en entrenamiento, las predicciones ahí serán extrapolaciones
y deben reportarse con mayor incertidumbre.

In [ ]:
if ctn is not None:
    rows = []
    for c in [c for c in num if c in ctn.columns]:
        a, b = train[c].dropna(), ctn[c].dropna()
        if len(a) > 5 and len(b) > 5:
            outside = ((b < a.min()) | (b > a.max())).mean() * 100
            rows.append([c, stats.ks_2samp(a, b).pvalue, outside])
    shift = pd.DataFrame(rows, columns=["feature", "KS p_value", "% CTN fuera de rango de entrenamiento"]).sort_values("KS p_value")
    display(shift.round(4).head(15))

## 6. Preprocesamiento (exploratorio; en modelado se ajusta sólo con entrenamiento dentro de cada fold)
### 6.1 Faltantes
| Caso | Estrategia | Justificación |
|---|---|---|
| Conteos (población, DENUE, localidades) = 0 | Se conservan | Cero real, no faltante |
| `mun_agri_*` sin registro SIAP | Indicador + mediana | Faltante estructural (municipios urbanos) |
| Variables de Censo en buffers sin localidades o con `*` | Indicador + mediana | Confidencialidad / sin población |
| Cualquier otra numérica | Mediana + indicador `_missing` | Robusta a sesgo; el indicador conserva la información de ausencia |

In [ ]:
pre = df.copy(); log = []
for c in [c for c in num if pre[c].isna().any()]:
    pre[f"{c}_missing"] = pre[c].isna().astype(int); n0 = pre[c].isna().sum()
    pre[c] = pre[c].fillna(pre.loc[~pre["is_corridor_test"], c].median())  # mediana SOLO de entrenamiento
    log.append([c, n0, "mediana (entrenamiento) + indicador"])
for c in [c for c in cat if pre[c].isna().any()]:
    n0 = pre[c].isna().sum(); pre[c] = pre[c].fillna("FALTANTE"); log.append([c, n0, "categoría FALTANTE"])
display(pd.DataFrame(log, columns=["variable", "n_imputados", "estrategia"]))

### 6.2 Atípicos
No se eliminan sitios: los TDPA extremos (zona metropolitana, libramientos de Cuernavaca) son reales.
Estrategia: objetivo en escala log, `log1p` en features sesgadas y modelos robustos (árboles) o escalado robusto.

In [ ]:
rows = []
for c in show:
    s = pre[c]; q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    k = ((s < q1 - 1.5*i) | (s > q3 + 1.5*i)).sum()
    if k: rows.append([c, k, round(100*k/len(s), 1)])
display(pd.DataFrame(rows, columns=["feature", "atípicos IQR", "%"]).sort_values("atípicos IQR", ascending=False))
for c in SKEWED: pre[f"log1p_{c}"] = np.log1p(pre[c].clip(lower=0))

### 6.3 Alta cardinalidad
- `group_route` (carretera) y `group_block25` **no son features**: se usan sólo para agrupar la validación.
- DENUE `Clase_actividad` → agregado a sectores SCIAN y conteos temáticos + diversidad de Shannon.
- Cultivos SIAP → número de cultivos, diversidad y cultivo principal; cultivos con < 5 sitios → `OTROS`.

In [ ]:
if "mun_agri_top_crop" in pre:
    vc = pre["mun_agri_top_crop"].value_counts(); rare = vc[vc < 5].index
    pre["mun_agri_top_crop_grp"] = pre["mun_agri_top_crop"].where(~pre["mun_agri_top_crop"].isin(rare), "OTROS")
    print("cultivo principal:", pre["mun_agri_top_crop"].nunique(), "->", pre["mun_agri_top_crop_grp"].nunique(), "categorías")

## 7. Selección preliminar de características (objetivo 2.1)
Criterios **no supervisados** (no usan el objetivo, por lo que no generan leakage): varianza casi nula y
redundancia |Spearman| > 0.90 entre features. La selección basada en relevancia con el objetivo queda para
el modelado, dentro de cada fold.

In [ ]:
cand = [c for c in show]
cv_ = pre[cand].std() / pre[cand].mean().abs().replace(0, np.nan)
low = cv_[cv_ < 0.01].index.tolist(); keep = [c for c in cand if c not in low]
sp2 = pre[keep].corr(method="spearman").abs(); compl = df[keep].notna().mean(); drop = set()
for i, a in enumerate(keep):
    for b in keep[i+1:]:
        if a not in drop and b not in drop and sp2.loc[a, b] > .90:
            drop.add(b if compl[a] >= compl[b] else a)
selected = [c for c in keep if c not in drop] + [c for c in cat]
sel = pd.DataFrame({"feature": cand + cat, "decisión": ["varianza ~0" if c in low else "redundante" if c in drop else "seleccionada" for c in cand + cat]})
display(sel.merge(prov[["feature_name", "class", "source"]], left_on="feature", right_on="feature_name", how="left").drop(columns="feature_name"))
print(f"{len(cand) + len(cat)} -> {len(selected)} features")
cols = ["site_id", "lon", "lat", "tdpa", TARGET, "group_state", "group_route", "group_block25", "is_corridor_test"] + selected + \
       [c for c in pre.columns if c.endswith("_missing") or c.startswith("log1p_") or c.endswith("_grp")]
pre[list(dict.fromkeys(cols))].to_csv(P / "TOP_Model_Table_preprocessed.csv", index=False)
sel.to_csv(ROOT / "06_RESULTS" / "feature_selection_top.csv", index=False)

## 8. Diseño de validación (vista previa para el modelado)

In [ ]:
from sklearn.model_selection import GroupKFold
gkf = GroupKFold(n_splits=5)
for gcol in ["group_block25", "group_route"]:
    sizes = [len(te) for _, te in gkf.split(train, groups=train[gcol])]
    print(f"GroupKFold por {gcol}: {train[gcol].nunique()} grupos -> tamaños de fold de prueba {sizes}")
print(f"Prueba externa (MEX-095D, nunca entrena): {len(test)} sitios")

**Plan para el Avance 2:** baseline ingenuo (mediana de ln(TDPA) por tipo de red y cuota), baseline Ridge,
y alternativos Random Forest y Gradient Boosting; métricas MAE/RMSE (veh/día) y R² (escala log) bajo
GroupKFold espacial vs. partición aleatoria (para mostrar la inflación), intervalos por regresión cuantílica o
conformal, y SHAP calculado dentro de los folds.

## 9. Conclusiones (completar con resultados reales)
- **Faltantes:** ¿qué fuentes y dónde? ¿la ausencia se relaciona con el objetivo (§1.4)?
- **Objetivo:** TDPA fuertemente sesgado; en escala log es aproximadamente simétrico (§2.1).
- **Grupos:** diferencias de ln(TDPA) por cuota/red/estado (§2.2) y representación de cada grupo.
- **Relaciones:** features más asociadas al objetivo y su dirección (§5.1); bloques redundantes (§5.2).
- **Autocorrelación espacial:** distancia a la que se pierde la correlación → tamaño de bloque (§5.3).
- **Aplicabilidad:** qué features de los CTN caen fuera del rango de entrenamiento (§5.4).
- **Preprocesamiento:** estrategias aplicadas y justificadas (§6); features retenidas (§7).
- **Limitaciones:** un solo estado de PDF (ampliar con CDMX, Edomex, Guerrero, Puebla); TDPA mezcla tipos de
  estación; contexto municipal ≠ predial; NASA POWER es malla regional; sin datos de capacidad eléctrica.